---

## 1️⃣ Importar Dependencias

In [1]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
import tensorflow as tf

print(f"TensorFlow version: {tf.__version__}")
print(f"OpenCV version: {cv2.__version__}")

2025-12-20 00:15:13.824122: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-20 00:15:14.578183: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-12-20 00:15:16.737657: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


TensorFlow version: 2.20.0
OpenCV version: 4.12.0


---

## 2️⃣ Definir Clases Custom (para cargar el modelo)

El modelo fue entrenado con custom losses, por lo que necesitamos definirlas para poder cargarlo.

In [2]:
# Custom Loss Classes
class SSDBoxLoss(keras.losses.Loss):
    """Loss for bounding box regression (Smooth L1)"""
    def call(self, y_true, y_pred):
        absolute_loss = tf.abs(y_true - y_pred)
        square_loss = 0.5 * (y_true - y_pred) ** 2
        loss = tf.where(tf.less(absolute_loss, 1.0), square_loss, absolute_loss - 0.5)
        return tf.reduce_sum(loss, axis=-1)

class SSDClassLoss(keras.losses.Loss):
    """Loss for classification (Cross Entropy)"""
    def call(self, y_true, y_pred):
        cross_entropy = tf.nn.softmax_cross_entropy_with_logits(labels=y_true, logits=y_pred)
        return cross_entropy

print("✅ Custom loss classes definidas")

✅ Custom loss classes definidas


---

## 3️⃣ Cargar Modelo Entrenado

Cargaremos el mejor modelo guardado durante el entrenamiento.

In [3]:
# Ruta al modelo
model_path = "./models/ssd_mobilenet_best.keras"

# Verificar que existe
if not os.path.exists(model_path):
    print(f"❌ ERROR: No se encuentra el modelo en {model_path}")
    print("Asegúrate de haber entrenado el modelo primero.")
else:
    # Cargar modelo con custom objects
    model = keras.models.load_model(
        model_path,
        custom_objects={
            'SSDBoxLoss': SSDBoxLoss,
            'SSDClassLoss': SSDClassLoss
        }
    )
    print(f"✅ Modelo cargado correctamente desde {model_path}")
    print(f"   Input shape: {model.input_shape}")
    print(f"   Output shape: {model.output_shape}")

TypeError: Could not locate class 'SSDModel'. Make sure custom classes and functions are decorated with `@keras.saving.register_keras_serializable()`. If they are already decorated, make sure they are all imported so that the decorator is run before trying to load them. Full object config: {'module': None, 'class_name': 'SSDModel', 'config': {'name': 'ssd_model_1', 'trainable': True, 'dtype': {'module': 'keras', 'class_name': 'DTypePolicy', 'config': {'name': 'float32'}, 'registered_name': None}}, 'registered_name': 'SSDModel', 'compile_config': {'optimizer': {'module': 'keras.optimizers', 'class_name': 'Adam', 'config': {'name': 'adam', 'learning_rate': 9.999999747378752e-05, 'weight_decay': None, 'clipnorm': None, 'global_clipnorm': None, 'clipvalue': None, 'use_ema': False, 'ema_momentum': 0.99, 'ema_overwrite_frequency': None, 'loss_scale_factor': None, 'gradient_accumulation_steps': None, 'beta_1': 0.9, 'beta_2': 0.999, 'epsilon': 1e-07, 'amsgrad': False}, 'registered_name': None}

---

## 4️⃣ Funciones de Procesamiento

Funciones para preprocesar imágenes y post-procesar las predicciones.

In [ ]:
def preprocess_image(image, target_size=(300, 300)):
    """
    Preprocesa imagen para el modelo.
    
    Args:
        image: Imagen BGR (OpenCV)
        target_size: Tamaño objetivo (ancho, alto)
    
    Returns:
        Imagen preprocesada normalizada
    """
    # Redimensionar
    img_resized = cv2.resize(image, target_size)
    
    # Convertir BGR a RGB
    img_rgb = cv2.cvtColor(img_resized, cv2.COLOR_BGR2RGB)
    
    # Normalizar [0, 255] -> [0, 1]
    img_normalized = img_rgb.astype(np.float32) / 255.0
    
    return img_normalized


def decode_predictions(predictions, confidence_threshold=0.5, iou_threshold=0.45):
    """
    Decodifica predicciones del modelo y aplica NMS.
    
    Args:
        predictions: Output del modelo [batch, num_anchors, 6]
                    Format: [x_center, y_center, width, height, class_prob, objectness]
        confidence_threshold: Umbral mínimo de confianza
        iou_threshold: Umbral IoU para NMS
    
    Returns:
        boxes: Lista de cajas [x1, y1, x2, y2]
        scores: Lista de scores de confianza
    """
    # Obtener primera imagen del batch
    pred = predictions[0]  # Shape: [num_anchors, 6]
    
    # Extraer componentes
    box_centers_x = pred[:, 0]
    box_centers_y = pred[:, 1]
    box_widths = pred[:, 2]
    box_heights = pred[:, 3]
    class_probs = pred[:, 4]  # Probabilidad clase 'car'
    objectness = pred[:, 5]   # Objectness score
    
    # Score final = class_prob * objectness
    scores = class_probs * objectness
    
    # Filtrar por confidence threshold
    mask = scores >= confidence_threshold
    
    if not np.any(mask):
        return [], []
    
    # Aplicar máscara
    box_centers_x = box_centers_x[mask]
    box_centers_y = box_centers_y[mask]
    box_widths = box_widths[mask]
    box_heights = box_heights[mask]
    scores = scores[mask]
    
    # Convertir de (cx, cy, w, h) a (x1, y1, x2, y2)
    x1 = box_centers_x - box_widths / 2
    y1 = box_centers_y - box_heights / 2
    x2 = box_centers_x + box_widths / 2
    y2 = box_centers_y + box_heights / 2
    
    # Asegurar que están en rango [0, 1]
    x1 = np.clip(x1, 0, 1)
    y1 = np.clip(y1, 0, 1)
    x2 = np.clip(x2, 0, 1)
    y2 = np.clip(y2, 0, 1)
    
    boxes = np.stack([x1, y1, x2, y2], axis=-1)
    
    # Aplicar NMS (Non-Maximum Suppression)
    indices = tf.image.non_max_suppression(
        boxes,
        scores,
        max_output_size=50,
        iou_threshold=iou_threshold,
        score_threshold=confidence_threshold
    ).numpy()
    
    final_boxes = boxes[indices].tolist()
    final_scores = scores[indices].tolist()
    
    return final_boxes, final_scores


def draw_boxes(image, boxes, scores):
    """
    Dibuja cajas en la imagen.
    
    Args:
        image: Imagen BGR (OpenCV)
        boxes: Lista de cajas normalizadas [x1, y1, x2, y2]
        scores: Lista de scores de confianza
    
    Returns:
        Imagen con cajas dibujadas
    """
    img_height, img_width = image.shape[:2]
    img_copy = image.copy()
    
    for box, score in zip(boxes, scores):
        # Convertir coordenadas normalizadas a píxeles
        x1 = int(box[0] * img_width)
        y1 = int(box[1] * img_height)
        x2 = int(box[2] * img_width)
        y2 = int(box[3] * img_height)
        
        # Dibujar rectángulo
        color = (0, 255, 0)  # Verde
        thickness = 2
        cv2.rectangle(img_copy, (x1, y1), (x2, y2), color, thickness)
        
        # Añadir texto con score
        label = f"Car: {score:.2f}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        font_thickness = 1
        
        # Fondo para el texto
        (text_width, text_height), _ = cv2.getTextSize(label, font, font_scale, font_thickness)
        cv2.rectangle(img_copy, (x1, y1 - text_height - 5), (x1 + text_width, y1), color, -1)
        
        # Texto
        cv2.putText(img_copy, label, (x1, y1 - 5), font, font_scale, (0, 0, 0), font_thickness)
    
    return img_copy

print("✅ Funciones de procesamiento definidas")

---

## 5️⃣ Predicción en Imagen

Prueba el modelo con una imagen de prueba.

In [ ]:
# Cargar imagen de prueba
test_image_path = "01b7349a9dc671014e6e3458db18c5a4.jpg"  # ⬅️ CAMBIA ESTA RUTA

# O puedes cargar desde webcam
# cap = cv2.VideoCapture(0)
# ret, image = cap.read()
# cap.release()

if os.path.exists(test_image_path):
    # Leer imagen
    image = cv2.imread(test_image_path)
    
    if image is not None:
        print(f"✅ Imagen cargada: {image.shape}")
        
        # Preprocesar
        img_preprocessed = preprocess_image(image)
        img_batch = np.expand_dims(img_preprocessed, axis=0)
        
        # Predicción
        print("🔮 Realizando predicción...")
        predictions = model.predict(img_batch, verbose=0)
        
        # Decodificar predicciones
        boxes, scores = decode_predictions(
            predictions,
            confidence_threshold=0.5,
            iou_threshold=0.45
        )
        
        print(f"✅ Detectados {len(boxes)} vehículos")
        
        # Dibujar cajas
        result_image = draw_boxes(image, boxes, scores)
        
        # Mostrar resultado
        plt.figure(figsize=(12, 8))
        plt.imshow(cv2.cvtColor(result_image, cv2.COLOR_BGR2RGB))
        plt.axis('off')
        plt.title(f"Detecciones: {len(boxes)} vehículos")
        plt.tight_layout()
        plt.show()
    else:
        print("❌ Error al cargar la imagen")
else:
    print(f"❌ No se encuentra el archivo: {test_image_path}")
    print("   Cambia la ruta 'test_image_path' a una imagen válida.")

---

## 6️⃣ Predicción en Tiempo Real (Webcam)

⚠️ **IMPORTANTE**: Este código debe ejecutarse fuera del notebook para mejor rendimiento.
Presiona `q` para salir.

In [ ]:
def run_webcam_detection(confidence_threshold=0.5, show_fps=True):
    """
    Ejecuta detección en tiempo real desde la webcam.
    
    Args:
        confidence_threshold: Umbral mínimo de confianza
        show_fps: Mostrar FPS en pantalla
    """
    # Abrir webcam
    cap = cv2.VideoCapture(0)
    
    if not cap.isOpened():
        print("❌ Error: No se pudo abrir la webcam")
        return
    
    print("✅ Webcam iniciada")
    print("📹 Presiona 'q' para salir")
    
    # Variables para FPS
    import time
    prev_time = time.time()
    
    try:
        while True:
            # Leer frame
            ret, frame = cap.read()
            
            if not ret:
                print("❌ Error al leer frame")
                break
            
            # Preprocesar
            img_preprocessed = preprocess_image(frame)
            img_batch = np.expand_dims(img_preprocessed, axis=0)
            
            # Predicción
            predictions = model.predict(img_batch, verbose=0)
            
            # Decodificar
            boxes, scores = decode_predictions(
                predictions,
                confidence_threshold=confidence_threshold,
                iou_threshold=0.45
            )
            
            # Dibujar cajas
            result_frame = draw_boxes(frame, boxes, scores)
            
            # Calcular FPS
            if show_fps:
                curr_time = time.time()
                fps = 1 / (curr_time - prev_time)
                prev_time = curr_time
                
                # Mostrar FPS y conteo
                cv2.putText(result_frame, f"FPS: {fps:.1f}", (10, 30),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                cv2.putText(result_frame, f"Cars: {len(boxes)}", (10, 70),
                           cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
            
            # Mostrar frame
            cv2.imshow('Vehicle Detection - Press Q to quit', result_frame)
            
            # Salir con 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
    
    finally:
        # Limpiar
        cap.release()
        cv2.destroyAllWindows()
        print("✅ Webcam cerrada")

# Ejecutar (descomentar para correr)
# run_webcam_detection(confidence_threshold=0.5)

---

## 7️⃣ Predicción en Video

Procesa un archivo de video y guarda el resultado.

In [ ]:
def process_video(input_path, output_path, confidence_threshold=0.5):
    """
    Procesa un video completo y guarda el resultado.
    
    Args:
        input_path: Ruta del video de entrada
        output_path: Ruta del video de salida
        confidence_threshold: Umbral mínimo de confianza
    """
    # Abrir video
    cap = cv2.VideoCapture(input_path)
    
    if not cap.isOpened():
        print(f"❌ Error: No se pudo abrir el video {input_path}")
        return
    
    # Obtener propiedades del video
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    
    print(f"📹 Video: {width}x{height} @ {fps} FPS")
    print(f"   Total frames: {total_frames}")
    
    # Crear VideoWriter
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))
    
    frame_count = 0
    
    try:
        while True:
            ret, frame = cap.read()
            
            if not ret:
                break
            
            frame_count += 1
            
            # Preprocesar
            img_preprocessed = preprocess_image(frame)
            img_batch = np.expand_dims(img_preprocessed, axis=0)
            
            # Predicción
            predictions = model.predict(img_batch, verbose=0)
            
            # Decodificar
            boxes, scores = decode_predictions(
                predictions,
                confidence_threshold=confidence_threshold,
                iou_threshold=0.45
            )
            
            # Dibujar cajas
            result_frame = draw_boxes(frame, boxes, scores)
            
            # Escribir frame
            out.write(result_frame)
            
            # Mostrar progreso
            if frame_count % 30 == 0:
                progress = (frame_count / total_frames) * 100
                print(f"\rProgreso: {progress:.1f}% ({frame_count}/{total_frames})", end="")
    
    finally:
        cap.release()
        out.release()
        print(f"\n✅ Video procesado guardado en: {output_path}")

# Ejemplo de uso (descomentar para correr)
# process_video(
#     input_path="ruta/al/video.mp4",
#     output_path="ruta/salida/video_detectado.mp4",
#     confidence_threshold=0.5
# )

---

## 8️⃣ Ajustar Parámetros de Detección

Puedes experimentar con diferentes umbrales:

In [ ]:
# Parámetros ajustables
CONFIDENCE_THRESHOLD = 0.5  # Mayor = menos detecciones pero más precisas
IOU_THRESHOLD = 0.45        # Mayor = menos supresión de cajas superpuestas

print("🎛️ Parámetros actuales:")
print(f"   Confidence threshold: {CONFIDENCE_THRESHOLD}")
print(f"   IoU threshold: {IOU_THRESHOLD}")
print("\n💡 Ajusta estos valores según tus necesidades:")
print("   - confidence_threshold entre 0.3-0.7 (típico: 0.5)")
print("   - iou_threshold entre 0.3-0.5 (típico: 0.45)")

---

## 📝 Notas Finales

### ✅ Funcionalidades Implementadas:
- ✅ Carga del modelo entrenado
- ✅ Detección en imágenes estáticas
- ✅ Detección en tiempo real (webcam)
- ✅ Procesamiento de videos
- ✅ Visualización de resultados

### 🎯 Próximos Pasos:
1. **Optimización**: Usar TensorRT o ONNX para mayor velocidad
2. **Tracking**: Implementar seguimiento de vehículos entre frames
3. **License Plates**: Agregar detector de matrículas como segunda etapa
4. **Analytics**: Contar vehículos, detectar violaciones de tráfico, etc.

### 🐛 Troubleshooting:
- **Modelo no carga**: Verifica que existe `models/ssd_mobilenet_best.keras`
- **Baja FPS**: Reduce resolución de entrada o usa GPU
- **Pocas detecciones**: Reduce `confidence_threshold`
- **Muchas detecciones falsas**: Aumenta `confidence_threshold`

---

**¡Listo para detectar vehículos! 🚗🎉**